# Data Mining Project - Phase 2
**Group #:** 5

**Members:** Adam Wajda, Bradley Krawchyk, Ekam Lally, Tanishk Sharma

## 1. Cluster Analysis

### Purpose and prepared data

This section groups households with similar grocery-shopping behaviour. Household-level clustering is more useful than clustering individual receipt lines because one row represents one shopper's overall pattern. The analysis uses the Phase 1 prepared table and only **standard merchandise lines**. Zero-quantity noise was already removed, and the special fuel/unit-of-measure lines are excluded so that their large recorded quantities do not distort distances.

The steps are: build household attributes, review correlations, select a smaller feature set, standardize the features, compare K-means (centroid) with Ward agglomerative clustering (hierarchy), and interpret the results.


In [ ]:
# Load the Phase 1 prepared data from the shared project folder.
from pathlib import Path
import sqlite3
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
RANDOM_SEED = 74030

# Run this notebook from the repository root so data/ resolves correctly.
root = Path.cwd()
if not (root / "data").is_dir():
    root = root.parent

PROJECT_DIR = root / "data"

DATABASE_PATH = PROJECT_DIR / 'complete_journey.sqlite'
if not DATABASE_PATH.exists():
    required_csvs = ['transactions_sample_part1a.csv', 'transactions_sample_part1b.csv',
                     'transactions_sample_part2.csv', 'product.csv']
    missing = [name for name in required_csvs if not (PROJECT_DIR / name).exists()]
    if missing:
        raise FileNotFoundError(f'Missing source data in {PROJECT_DIR}: {missing}. See DATA_SETUP.md')

if DATABASE_PATH.exists():
    with sqlite3.connect(DATABASE_PATH) as connection:
        cluster_lines = pd.read_sql_query(
            """
            SELECT household_key, basket_id, product_id, quantity, sales_value,
                   retail_disc, coupon_disc, coupon_match_disc, department
            FROM analysis_transactions_prepared
            WHERE line_type = 'Standard merchandise line'
            """,
            connection,
        )
else:
    # Rebuild the same standard-line subset from the source CSVs when testing outside Colab.
    part_names = [
        'transactions_sample_part1a.csv',
        'transactions_sample_part1b.csv',
        'transactions_sample_part2.csv',
    ]
    transactions = pd.concat(
        [pd.read_csv(PROJECT_DIR / name) for name in part_names],
        ignore_index=True,
    ).rename(columns=str.lower)
    products = pd.read_csv(
        PROJECT_DIR / 'product.csv', keep_default_na=False
    ).rename(columns=str.lower)
    products['department'] = (
        products['department'].astype(str).str.strip().replace('', 'Not specified')
    )
    cluster_lines = transactions.merge(
        products[['product_id', 'department']], on='product_id', how='left'
    )
    cluster_lines['department'] = cluster_lines['department'].fillna('Not specified')
    cluster_lines = cluster_lines.loc[
        (cluster_lines['quantity'] > 0) & (cluster_lines['quantity'] <= 100)
    ].reset_index(drop=True)

currency_columns = ['sales_value', 'retail_disc', 'coupon_disc', 'coupon_match_disc']
cluster_lines[currency_columns] = cluster_lines[currency_columns].round(2)
cluster_lines['discount_amount'] = (
    -cluster_lines[['retail_disc', 'coupon_disc', 'coupon_match_disc']].sum(axis=1)
).clip(lower=0)

print(f'Standard merchandise lines: {len(cluster_lines):,}')
print(f'Households available:       {cluster_lines["household_key"].nunique():,}')


### Household attributes

Eight candidate attributes are created from the prepared receipt lines:

- `basket_count`: number of shopping trips.
- `purchase_lines`: number of receipt lines.
- `total_spend`: total recorded sales value.
- `avg_basket_value`: typical dollars spent per trip.
- `avg_basket_items`: typical number of items per trip.
- `discount_share`: discount amount divided by value before discounts.
- `unique_products`: number of distinct products purchased.
- `unique_departments`: breadth of departments visited.

`household_key` is kept only as the row index. It is an identifier, not a behavioural attribute, so it is not used in the correlation or clustering.


In [ ]:
# First summarize each basket, then summarize each household.
basket_summary = (
    cluster_lines.groupby(['household_key', 'basket_id'])
    .agg(
        basket_sales=('sales_value', 'sum'),
        basket_items=('quantity', 'sum'),
        basket_discount=('discount_amount', 'sum'),
    )
    .reset_index()
)

household_features = (
    cluster_lines.groupby('household_key')
    .agg(
        basket_count=('basket_id', 'nunique'),
        purchase_lines=('product_id', 'size'),
        total_spend=('sales_value', 'sum'),
        total_discount=('discount_amount', 'sum'),
        unique_products=('product_id', 'nunique'),
        unique_departments=('department', 'nunique'),
    )
    .join(
        basket_summary.groupby('household_key').agg(
            avg_basket_value=('basket_sales', 'mean'),
            avg_basket_items=('basket_items', 'mean'),
        )
    )
)

household_features['discount_share'] = (
    household_features['total_discount']
    .div(household_features['total_spend'] + household_features['total_discount'])
    .fillna(0)
)

candidate_features = [
    'basket_count',
    'purchase_lines',
    'total_spend',
    'avg_basket_value',
    'avg_basket_items',
    'discount_share',
    'unique_products',
    'unique_departments',
]
household_features = household_features[candidate_features]

print(f'Household analysis table: {len(household_features):,} rows x {household_features.shape[1]} attributes')
display(household_features.describe().T.round(2))


### Correlation analysis

Pearson correlation is used because all eight candidate attributes are numeric. The heatmap includes **every attribute in the household analysis table** and makes redundant measurements easy to see.


In [ ]:
correlation = household_features.corr()

plt.figure(figsize=(11, 8))
sns.heatmap(
    correlation,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    linewidths=0.5,
)
plt.title('Correlation of All Household Analysis Attributes')
plt.tight_layout()
plt.show()

# List only very strong pairs so the feature decision is easy to check.
upper_triangle = correlation.where(
    np.triu(np.ones(correlation.shape), k=1).astype(bool)
)
strong_pairs = (
    upper_triangle.stack()
    .rename('correlation')
    .reset_index()
    .rename(columns={'level_0': 'attribute_1', 'level_1': 'attribute_2'})
)
strong_pairs = strong_pairs.loc[
    strong_pairs['correlation'].abs() >= 0.85
].sort_values('correlation', key=abs, ascending=False)

print('Very strong relationships (absolute correlation >= 0.85):')
display(strong_pairs.round(2))


### Feature analysis and selected subset

The heatmap shows four strong redundancies: receipt lines and unique products are almost identical ((r=0.99)); both also track total spend ((r=0.95)); and average basket value closely follows average basket items ((r=0.92)). If all of these were used, shopping activity and basket size would be counted several times in the distance calculation.

| Candidate attribute | Decision | Reason |
|---|---|---|
| `basket_count` | Drop | Overall activity is already represented by total spend; keeping both would give activity extra weight. |
| `purchase_lines` | Drop | It is almost the same signal as unique products and is strongly related to total spend. |
| `total_spend` | **Keep** | Direct measure of the household's overall value/activity. |
| `avg_basket_value` | Drop | Strongly overlaps with average basket items; trip size is easier to interpret beside total spend. |
| `avg_basket_items` | **Keep** | Represents the size of a typical grocery trip. |
| `discount_share` | **Keep** | Promotion sensitivity is conceptually important and has little correlation with the other attributes. |
| `unique_products` | Drop | Strongly duplicates purchase lines and total spend. |
| `unique_departments` | **Keep** | Measures shopping breadth and is less redundant than product count. |

The final subset is therefore **total spend, average basket items, discount share, and unique departments**. This gives one feature for customer value, trip size, promotion use, and product breadth. Skewed count and money features are log-transformed, then all four features are standardized so that dollars do not dominate the distance calculations.


In [ ]:
selected_features = [
    'total_spend',
    'avg_basket_items',
    'discount_share',
    'unique_departments',
]

model_features = household_features[selected_features].copy()
log_features = ['total_spend', 'avg_basket_items', 'unique_departments']
model_features[log_features] = np.log1p(model_features[log_features])

scaler = StandardScaler()
X_scaled = scaler.fit_transform(model_features)

scaled_summary = pd.DataFrame(
    X_scaled, columns=selected_features, index=model_features.index
).agg(['mean', 'std']).round(2)
display(scaled_summary)


### Method 1: K-means clustering

K-means is a centroid method. It is simple and works well after standardization. Values from (k=2) through (k=6) are checked. The main selection measure is silhouette score, where a larger value means that households are more similar to their own cluster and better separated from other clusters.


In [ ]:
k_rows = []
for k in range(2, 7):
    model = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_SEED)
    labels = model.fit_predict(X_scaled)
    k_rows.append({
        'k': k,
        'silhouette_score': silhouette_score(X_scaled, labels),
        'inertia': model.inertia_,
    })

k_results = pd.DataFrame(k_rows)
best_k = int(k_results.loc[k_results['silhouette_score'].idxmax(), 'k'])

figure, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.lineplot(data=k_results, x='k', y='silhouette_score', marker='o', ax=axes[0])
axes[0].set_title('K-means Silhouette Score')
axes[0].set_ylabel('Silhouette score')
axes[0].set_xticks(k_results['k'])

sns.lineplot(data=k_results, x='k', y='inertia', marker='o', ax=axes[1])
axes[1].set_title('K-means Elbow Plot')
axes[1].set_ylabel('Inertia')
axes[1].set_xticks(k_results['k'])

plt.tight_layout()
plt.show()
display(k_results.round(3))
print(f'Selected number of clusters: k = {best_k}')


### Method 2: Ward agglomerative clustering

Agglomerative clustering is a hierarchy method. It begins with each household as its own group and repeatedly merges nearby groups. Ward linkage is used because it tries to keep the variation inside each cluster small. It uses the same standardized features and the same number of clusters as K-means, which makes the comparison fair.


In [ ]:
# Fit both methods.
kmeans = KMeans(n_clusters=best_k, n_init=10, random_state=RANDOM_SEED)
kmeans_labels = kmeans.fit_predict(X_scaled)

hierarchical = AgglomerativeClustering(n_clusters=best_k, linkage='ward')
hierarchical_labels = hierarchical.fit_predict(X_scaled)

# Re-number clusters from lower to higher average spending for consistent interpretation.
def order_by_spending(labels):
    mean_spend = (
        household_features.assign(cluster=labels)
        .groupby('cluster')['total_spend']
        .mean()
        .sort_values()
    )
    mapping = {old_label: new_label for new_label, old_label in enumerate(mean_spend.index)}
    return np.array([mapping[label] for label in labels])

kmeans_labels = order_by_spending(kmeans_labels)
hierarchical_labels = order_by_spending(hierarchical_labels)

method_comparison = pd.DataFrame({
    'method': ['K-means', 'Ward hierarchical'],
    'clusters': [best_k, best_k],
    'silhouette_score': [
        silhouette_score(X_scaled, kmeans_labels),
        silhouette_score(X_scaled, hierarchical_labels),
    ],
})
agreement = adjusted_rand_score(kmeans_labels, hierarchical_labels)

display(method_comparison.round(3))
print(f'Adjusted Rand agreement between methods: {agreement:.3f}')


In [ ]:
# PCA reduces four dimensions to two only for visualization.
pca = PCA(n_components=2)
plot_points = pca.fit_transform(X_scaled)

figure, axes = plt.subplots(1, 2, figsize=(13, 5))
for axis, labels, title in zip(
    axes,
    [kmeans_labels, hierarchical_labels],
    ['K-means clusters', 'Ward hierarchical clusters'],
):
    sns.scatterplot(
        x=plot_points[:, 0],
        y=plot_points[:, 1],
        hue=labels,
        palette='Set1',
        s=35,
        alpha=0.70,
        linewidth=0,
        ax=axis,
    )
    axis.set_title(title)
    axis.set_xlabel('PCA component 1')
    axis.set_ylabel('PCA component 2')
    axis.legend(title='Cluster')

figure.suptitle(
    f'Two-dimensional View of Household Clusters '
    f'({pca.explained_variance_ratio_.sum():.1%} of variance shown)',
    y=1.03,
)
plt.tight_layout()
plt.show()

profile_columns = [
    'basket_count',
    'total_spend',
    'avg_basket_value',
    'avg_basket_items',
    'discount_share',
    'unique_departments',
]

def cluster_profile(labels):
    return (
        household_features.assign(cluster=labels)
        .groupby('cluster')
        .agg(
            households=('total_spend', 'size'),
            basket_count=('basket_count', 'mean'),
            total_spend=('total_spend', 'mean'),
            avg_basket_value=('avg_basket_value', 'mean'),
            avg_basket_items=('avg_basket_items', 'mean'),
            discount_share=('discount_share', 'mean'),
            unique_departments=('unique_departments', 'mean'),
        )
    )

kmeans_profile = cluster_profile(kmeans_labels)
hierarchical_profile = cluster_profile(hierarchical_labels)

print('K-means cluster profile (means):')
display(kmeans_profile.round(2))
print('Ward hierarchical cluster profile (means):')
display(hierarchical_profile.round(2))

# Standardized profile heatmaps show which attributes are high or low relative to all households.
overall_mean = household_features[profile_columns].mean()
overall_std = household_features[profile_columns].std()

figure, axes = plt.subplots(1, 2, figsize=(14, 4))
for axis, profile, title in zip(
    axes,
    [kmeans_profile, hierarchical_profile],
    ['K-means profile', 'Ward hierarchical profile'],
):
    standardized_profile = (
        profile[profile_columns].subtract(overall_mean).divide(overall_std)
    )
    sns.heatmap(
        standardized_profile,
        annot=True,
        fmt='.2f',
        cmap='coolwarm',
        center=0,
        ax=axis,
        cbar=False,
    )
    axis.set_title(title)
    axis.set_ylabel('Cluster (0 = lower spend)')
    axis.set_xlabel('Attribute')

plt.tight_layout()
plt.show()


### Discussion of clustering results

Both methods selected a clear two-group solution. K-means produced a silhouette score of **0.382**, while Ward hierarchical clustering scored **0.384**. These scores are moderate rather than excellent, so the groups are useful broad segments but are not perfectly separated. The adjusted Rand score of **0.767** shows that the two different methods placed most households into similar groups, which adds confidence that the pattern is not caused by only one algorithm.

For K-means, the lower-activity cluster contains **754 households**. Its households spent about **\$45.86**, bought about **6.31 items per basket**, and visited about **3.25 departments** on average. The higher-activity cluster contains **1,557 households**, with about **\$328.35** in total spending, **18.50 items per basket**, and **7.50 departments**. Ward clustering found nearly the same pattern, with **648 lower-activity** and **1,663 higher-activity** households. Its average spending was about **\$40.27** and **$312.52**, respectively.

The main separation is therefore customer activity, basket size, and shopping breadth. Discount share changes only slightly (about **14% to 16%**), so promotion use does not define the clusters as strongly as spending and trip size. A retailer could use the higher-activity group for broad cross-selling or loyalty offers, while the lower-activity group may respond better to simple offers intended to increase basket size. Because this is observational sample data and the clusters overlap, these are descriptive segments rather than proof that a promotion will cause different behaviour.


## 2. Decision Tree Analysis

### Purpose and target

A CART decision tree is used to identify shopping behaviours associated with higher-spending households. The target is created from `total_spend`: households spending at or above the median are labelled **Higher Spend**, while households below the median are labelled **Lower Spend**.

The predictors are `basket_count`, `avg_basket_items`, `discount_share`, and `unique_departments`. `total_spend` itself is excluded from the predictors because it defines the target. Highly related spending measures such as `avg_basket_value`, `purchase_lines`, and `unique_products` are also excluded to reduce redundancy and avoid making the prediction artificially easy.

The data is divided into 80% training and 20% testing data. Tree depth is evaluated using 5-fold cross-validation, and a shallow maximum depth is used as a pre-pruning strategy to reduce overfitting and keep the final decision tree legible.

In [ ]:
# Imports for decision tree analysis.
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)

# Create a meaningful binary target using median household spending.
spend_threshold = household_features['total_spend'].median()

tree_data = household_features.copy()
tree_data['high_spend'] = (
    tree_data['total_spend'] >= spend_threshold
).astype(int)

# Use behavioural attributes that do not directly define total spending.
tree_features = [
    'basket_count',
    'avg_basket_items',
    'discount_share',
    'unique_departments',
]

X_tree = tree_data[tree_features]
y_tree = tree_data['high_spend']

print(f'Median spending threshold: ${spend_threshold:.2f}')
print(f'Households used: {len(tree_data):,}')

print('\nTarget distribution:')
display(
    y_tree.value_counts()
    .rename(index={0: 'Lower Spend', 1: 'Higher Spend'})
    .to_frame('households')
)

display(X_tree.describe().round(2))

### Training and pruning strategy

The household data is split into training and testing sets using an 80-20 split. Because an unrestricted decision tree can continue creating branches around small groups of observations, it may overfit the training data.

Pre-pruning is therefore applied by controlling `max_depth`. Candidate depths from 2 through 6 are compared using 5-fold cross-validation on the training data. The final depth is selected by considering both validation accuracy and tree interpretability. Although depth 6 produced the highest validation accuracy, depth 5 achieved nearly identical performance while producing a simpler and more legible tree. A minimum leaf size is also used so that individual decisions are not based on very small groups of households.

In [ ]:
# 80-20 train/test split.
X_train, X_test, y_train, y_test = train_test_split(
    X_tree,
    y_tree,
    test_size=0.20,
    random_state=RANDOM_SEED,
    stratify=y_tree,
)

print(f'Training households: {len(X_train):,}')
print(f'Testing households:  {len(X_test):,}')

# Evaluate several shallow tree depths using 5-fold cross-validation.
depth_results = []

for depth in range(2, 7):
    model = DecisionTreeClassifier(
        criterion='gini',
        max_depth=depth,
        min_samples_leaf=25,
        random_state=RANDOM_SEED,
    )

    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=5,
        scoring='accuracy',
    )

    depth_results.append({
        'max_depth': depth,
        'mean_cv_accuracy': scores.mean(),
        'std_cv_accuracy': scores.std(),
    })

depth_results = pd.DataFrame(depth_results)

# Depth 5 is selected because its validation accuracy is nearly
# identical to depth 6 while producing a simpler, more legible tree.
best_depth = 5

display(depth_results.round(3))

print(f'Selected maximum depth: {best_depth}')

### Visualize the pruning decision

In [ ]:
plt.figure(figsize=(8, 4))

sns.lineplot(
    data=depth_results,
    x='max_depth',
    y='mean_cv_accuracy',
    marker='o',
)

plt.title('Decision Tree Depth Selection')
plt.xlabel('Maximum Tree Depth')
plt.ylabel('5-Fold Cross-Validation Accuracy')
plt.xticks(depth_results['max_depth'])
plt.ylim(0, 1)

plt.tight_layout()
plt.show()

### Final CART decision tree

The final CART classification tree uses the selected maximum depth and the Gini criterion. The model is trained using the training households and then evaluated on the previously unseen test households.

### Train and evaluate the final tree

In [ ]:
decision_tree = DecisionTreeClassifier(
    criterion='gini',
    max_depth=best_depth,
    min_samples_leaf=25,
    random_state=RANDOM_SEED,
)

decision_tree.fit(X_train, y_train)

y_pred = decision_tree.predict(X_test)

test_accuracy = accuracy_score(y_test, y_pred)

print(f'Test accuracy: {test_accuracy:.3f}')
print(f'Tree depth:    {decision_tree.get_depth()}')
print(f'Leaf nodes:    {decision_tree.get_n_leaves()}')

print('\nClassification report:')
print(
    classification_report(
        y_test,
        y_pred,
        target_names=['Lower Spend', 'Higher Spend'],
    )
)

confusion = pd.DataFrame(
    confusion_matrix(y_test, y_pred),
    index=['Actual Lower Spend', 'Actual Higher Spend'],
    columns=['Predicted Lower Spend', 'Predicted Higher Spend'],
)

print('Confusion matrix:')
display(confusion)

### Decision tree visualization

The pruned tree below shows the sequence of behavioural rules used to classify households. Each internal node represents a test on a household attribute, each branch represents the result of that test, and each leaf represents the predicted spending class.

In [ ]:
plt.figure(figsize=(22, 10))

plot_tree(
    decision_tree,
    feature_names=tree_features,
    class_names=['Lower Spend', 'Higher Spend'],
    filled=True,
    rounded=True,
    proportion=True,
    precision=2,
    fontsize=9,
)

plt.title(
    f'Pruned CART Decision Tree for Household Spending '
    f'(Maximum Depth = {best_depth})',
    fontsize=16,
)

plt.tight_layout()
plt.show()

### Feature importance

In [ ]:
feature_importance = pd.DataFrame({
    'feature': tree_features,
    'importance': decision_tree.feature_importances_,
}).sort_values('importance', ascending=False)

display(feature_importance.round(3))

plt.figure(figsize=(8, 4))

sns.barplot(
    data=feature_importance,
    x='importance',
    y='feature',
)

plt.title('Decision Tree Feature Importance')
plt.xlabel('Importance')
plt.ylabel('Household Attribute')

plt.tight_layout()
plt.show()

### Discussion of decision tree results

The CART decision tree was developed to classify households as higher or lower spenders using shopping behaviour rather than total spending itself. A median spending threshold of **$155.75** created two nearly equal classes, providing a balanced classification problem.

Tree depth was evaluated using 5-fold cross-validation. Increasing the maximum depth improved validation accuracy from **0.820 at depth 2** to **0.890 at depth 6**. However, depth 5 achieved a nearly identical accuracy of **0.889** while producing a simpler and more readable tree. Therefore, a maximum depth of 5 was selected as a pre-pruning strategy to reduce model complexity and limit overfitting.

The final decision tree showed that **unique departments** was the most influential predictor of household spending level, followed by **average basket items** and **basket count**. This suggests that higher-spending households are mainly distinguished by shopping across more departments, purchasing more items per basket, and making more shopping trips. In comparison, **discount share** had very little importance, suggesting that promotional purchasing behaviour was not a major factor in distinguishing higher- and lower-spending households.

Overall, the decision tree provides an interpretable set of behavioural rules for identifying higher-spending households. These results could help a retailer identify customers with broader and more active shopping patterns for loyalty programs, cross-selling, or targeted promotions. However, the model identifies associations in the observed data and does not show that any individual shopping behaviour directly causes higher spending.

## 3. Association Rules

### Purpose and item definition

This section finds products that are often bought together, which helps the cross-selling and promotional aims of the project. The analysis uses the Phase 1 prepared table and only **standard merchandise lines**, so fuel and unit-of-measure lines cannot appear in any rule.

Items are defined at the **commodity** level rather than the individual `product_id`. There are over 90,000 products but only 293 commodities, so product-level rules would be far too sparse to reach any usable support. Commodity level also matches the business question, because a retailer plans promotions around categories such as pasta sauce rather than one specific jar.

A basket is one shopping trip (`basket_id`). Baskets containing only one commodity are removed.

In [ ]:
# Dependencies are installed using requirements.txt at the repository root.


In [ ]:
from mlxtend.frequent_patterns import apriori, association_rules
import squarify

if DATABASE_PATH.exists():
    with sqlite3.connect(DATABASE_PATH) as connection:
        rule_lines = pd.read_sql_query(
            """
            SELECT basket_id, commodity_desc
            FROM analysis_transactions_prepared
            WHERE line_type = 'Standard merchandise line'
            """,
            connection,
        )
else:
    rule_lines = cluster_lines.merge(
        pd.read_csv(PROJECT_DIR / 'product.csv', keep_default_na=False)
        .rename(columns=str.lower)[['product_id', 'commodity_desc']],
        on='product_id',
        how='left',
    )[['basket_id', 'commodity_desc']]

rule_lines['commodity_desc'] = rule_lines['commodity_desc'].str.strip().str.title()

# Only baskets holding at least two different commodities.
commodities_per_basket = rule_lines.groupby('basket_id')['commodity_desc'].nunique()
multi_item_baskets = commodities_per_basket[commodities_per_basket >= 2].index
rule_lines = rule_lines[rule_lines['basket_id'].isin(multi_item_baskets)]

print(f'Baskets before filtering:     {commodities_per_basket.size:,}')
print(f'Single-commodity baskets:     {(commodities_per_basket < 2).sum():,}')
print(f'Baskets used for rules:       {rule_lines["basket_id"].nunique():,}')
print(f'Distinct commodities:         {rule_lines["commodity_desc"].nunique():,}')
print(f'Median commodities per basket: {commodities_per_basket[multi_item_baskets].median():.0f}')

### Item frequency and Tree Map

Frequency counts show how often each commodity appears in a basket. The Tree Map sizes each rectangle by that count, so the largest tiles are the commodities that drive most of the traffic. This also sets expectations for the rules that follow, because a very common item will produce high-support rules while a rarer item can only appear in a rule with low support.

In [ ]:
# Frequency counts
basket_total = rule_lines['basket_id'].nunique()

item_frequency = (
    rule_lines.groupby('commodity_desc')['basket_id']
    .nunique()
    .sort_values(ascending=False)
    .to_frame('baskets')
)
item_frequency['support'] = (item_frequency['baskets'] / basket_total).round(4)

display(item_frequency.head(15))

# Tree Map of the 25 most frequent
treemap_data = item_frequency.head(25).reset_index()
treemap_labels = [
    f'{row.commodity_desc}\n{row.support:.1%}'
    for row in treemap_data.itertuples()
]

plt.figure(figsize=(14, 7))

squarify.plot(
    sizes=treemap_data['baskets'],
    label=treemap_labels,
    color=sns.color_palette('Blues_r', len(treemap_data)),
    alpha=0.85,
    text_kwargs={'fontsize': 8},
    pad=True,
)

plt.title('Tree Map of the 25 Most Frequent Commodities (share of baskets)')
plt.axis('off')

plt.tight_layout()
plt.show()

The Tree Map is dominated by staples. Fluid milk, soft drinks, and bread each appear in roughly a third of baskets and anything below shrinks even more. This matters for the threshold, because even the most common commodity only has about 33% support, any pair of items will have support even lower than that, so a minimum support level in the low single digits is the choice.

### Frequent itemsets with Apriori

Apriori builds frequent itemsets from the bottom up. It counts single items, drops the ones below the minimum support, then combines the rest into pairs, and so on.  It stops at three items, longer sets are harder to act on and more expensive to compute.

Minimum support is **0.02**, meaning an itemset must appear in at least 2% of baskets (about 300). This is low enough to find items beyond the top staples, but high enough to ensure each itemset is based on real shopping trips, not random chance.

In [ ]:
# One row per basket, one column per commodity, True when present
basket_matrix = pd.crosstab(rule_lines['basket_id'], rule_lines['commodity_desc']) > 0

MIN_SUPPORT = 0.02

frequent_itemsets = apriori(
    basket_matrix,
    min_support=MIN_SUPPORT,
    use_colnames=True,
    max_len=3,
)
frequent_itemsets['items'] = frequent_itemsets['itemsets'].apply(len)

print(f'Basket matrix: {basket_matrix.shape[0]:,} baskets x {basket_matrix.shape[1]} commodities')
print(f'Frequent itemsets found at {MIN_SUPPORT:.0%} minimum support: {len(frequent_itemsets):,}')

display(
    frequent_itemsets.groupby('items')['support']
    .agg(['count', 'min', 'max'])
    .rename(columns={'count': 'itemsets'})
    .round(3)
)

In [ ]:
# Generate rules, then sort by each of the three metrics
association_results = association_rules(
    frequent_itemsets,
    metric='confidence',
    min_threshold=0.20,
)


def format_itemset(itemset):
    return ', '.join(sorted(itemset))


association_results['antecedents'] = association_results['antecedents'].apply(format_itemset)
association_results['consequents'] = association_results['consequents'].apply(format_itemset)

RULE_COLUMNS = ['antecedents', 'consequents', 'support', 'confidence', 'lift']

print(f'Candidate rules generated: {len(association_results):,}')
display(association_results[['support', 'confidence', 'lift']].describe().round(3))

print('\nTop 10 rules sorted by SUPPORT:')
display(association_results.nlargest(10, 'support')[RULE_COLUMNS].round(3))

print('Top 10 rules sorted by CONFIDENCE:')
display(association_results.nlargest(10, 'confidence')[RULE_COLUMNS].round(3))

print('Top 10 rules sorted by LIFT:')
display(association_results.nlargest(10, 'lift')[RULE_COLUMNS].round(3))

No single metric does good enough job alone:

* **Support** ranks by frequency. Top rules are staples like milk and bread, frequent but obvious, since those items are in almost every basket.
* **Confidence** ranks by how reliably the second item follows the first. It favours common occurances, so a rule can score high just because it has milk that everyon buys.
* **Lift** ranks by how much stronger the pairing is than chance. It finds genuine links (e.g., pasta sauce with pasta), but alone it can promote rules based on very few baskets.

A useful promotion rule must score well on all three, so thresholds are applied together.

### Selecting thresholds

The table below counts how many rules survive different combinations, so the thresholds can be chosen from evidence and not random guessed.

In [ ]:
# How many rules survive different threshold combinations.
sensitivity_rows = []
for support_level in [0.02, 0.03, 0.04]:
    for confidence_level in [0.30, 0.35, 0.45]:
        for lift_level in [1.2, 1.5, 2.0]:
            surviving = (
                (association_results['support'] >= support_level)
                & (association_results['confidence'] >= confidence_level)
                & (association_results['lift'] >= lift_level)
            ).sum()
            sensitivity_rows.append({
                'min_support': support_level,
                'min_confidence': confidence_level,
                'min_lift': lift_level,
                'rules': int(surviving),
            })

sensitivity = pd.DataFrame(sensitivity_rows).pivot_table(
    index=['min_support', 'min_confidence'],
    columns='min_lift',
    values='rules',
)
display(sensitivity)

Each threshold answers a different question, and each is set with a business reason:

* **Support at least 0.03.** The rule must appear in at least 3% of baskets (about 450 trips). This is higher than the 2% used for mining, because a promotion needs to reach enough customers to be worthwhile.
* **Confidence at least 0.35.** At least 35% of baskets with the first item also contain the second. Since the most common single item only reaches about 33%, anything above 35% already outperforms just pushing that top item.
* **Lift at least 1.5.** The pairing must be at least 50% more likely than random. Lift of 1.0 means no relationship, so a clear margin is needed to confirm the pattern is real, not just shared popularity.

The sensitivity table shows that these are a deliberate middle ground. Dropping support to 0.02 gives over a thousand rules, too many. Raising support to 0.04 and lift to 2.0 leaves only about fifty rules, cutting useful pairings. The chosen balance keeps the set big enough for exploration but small enough for review.

In [ ]:
MIN_SUPPORT_RULE = 0.03
MIN_CONFIDENCE = 0.35
MIN_LIFT = 1.5

interesting_rules = association_results[
    (association_results['support'] >= MIN_SUPPORT_RULE)
    & (association_results['confidence'] >= MIN_CONFIDENCE)
    & (association_results['lift'] >= MIN_LIFT)
].sort_values('lift', ascending=False).reset_index(drop=True)

print(f'Candidate rules:   {len(association_results):,}')
print(f'Interesting rules: {len(interesting_rules):,}')
print(f'Thresholds applied: support >= {MIN_SUPPORT_RULE}, '
      f'confidence >= {MIN_CONFIDENCE}, lift >= {MIN_LIFT}')

print('\nTop 15 interesting rules by lift:')
display(interesting_rules.head(15)[RULE_COLUMNS].round(3))

In [ ]:
# Surviving rules
plt.figure(figsize=(9, 5))

scatter = plt.scatter(
    association_results['support'],
    association_results['confidence'],
    c=association_results['lift'],
    cmap='viridis',
    alpha=0.5,
    s=18,
)

plt.colorbar(scatter, label='Lift')
plt.axhline(MIN_CONFIDENCE, color='red', linestyle='--', linewidth=1)
plt.axvline(MIN_SUPPORT_RULE, color='red', linestyle='--', linewidth=1)

plt.title('Association Rules by Support and Confidence (colour = lift)')
plt.xlabel('Support')
plt.ylabel('Confidence')

plt.tight_layout()
plt.show()

# Strongest pairings chart.
top_rules = interesting_rules.head(10).copy()
top_rules['rule'] = top_rules['antecedents'] + '  ->  ' + top_rules['consequents']

plt.figure(figsize=(9, 5))

sns.barplot(data=top_rules, x='lift', y='rule')

plt.title('Ten Strongest Interesting Rules by Lift')
plt.xlabel('Lift')
plt.ylabel('Rule')

plt.tight_layout()
plt.show()

### Discussion of association rule results

Apriori found **1,013 frequent itemsets** and **1,977 candidate rules**. Using all three thresholds together turned that number to **322 interesting rules**, about 1 in 6.

Top rules reflect natural meal pairings. Pasta sauce with pasta has lift **7.81**, and cheeses with deli meats have lift **7.37**, both with confidence above 0.40 and support over 3%. Apples with tropical fruit **3.32** and breakfast items with cereal **3.48** follow the same pattern.

The highest candidate lift **8.54**, bread with cheeses and deli meats was filtered out by the support threshold, strong, but too rare to make a promotion around it.

The scatter plot confirms the need for all three metrics: high-support rules lean on staples and have lower lift, while highest-lift rules sit at lower support. The thresholds select rules that are balanced across all three, not extreme on any one.

For the seller, these suggest direct actions: place pasta sauce with pasta, or cheeses with deli meats. Rules with staples like milk or bread as the outcome are less useful, those items are already found everywhere in many baskets, so promotions are unlikely to change behavior.

The main limitation is that these rules describe co-occurrence, not cause. Two items appearing together does not prove that promoting one will lift sales of the other, so any rule acted on should be confirmed as well. Commodity level also matters: a finer product level would produce more specific rules but far lower support, while a lower department level would produce rules too broad to act on.

## 4. Classification Analysis

### Purpose, target, and experimental design

The classification task predicts whether a household is a **Higher Spend** or **Lower Spend** household. The outcome is the same median-based `high_spend` target used in the decision-tree analysis, which makes the result meaningful to the grocery-retail domain and keeps the two sections comparable.

The predictors are `basket_count`, `avg_basket_items`, `discount_share`, and `unique_departments`. `total_spend` is deliberately excluded because it directly defines the target; including it would cause target leakage. A stratified 80-20 train/test split preserves the class balance. The 80% training partition is then divided into five stratified folds: each model trains on four folds and validates on the fifth, rotating until every fold has served once as validation data. The untouched 20% test partition is used only for the final comparison.


In [ ]:
# Three classifiers with different learning assumptions.
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

# Reuse the leakage-safe data and 80-20 split created above.
X_classification = X_tree.copy()
y_classification = y_tree.copy()

X_cls_train, X_cls_test, y_cls_train, y_cls_test = train_test_split(
    X_classification,
    y_classification,
    test_size=0.20,
    random_state=RANDOM_SEED,
    stratify=y_classification,
)

classifiers = {
    'Gaussian Naive Bayes': GaussianNB(),
    'K-Nearest Neighbours': Pipeline([
        ('scale', StandardScaler()),
        ('model', KNeighborsClassifier(n_neighbors=15, weights='distance')),
    ]),
    'RBF Support Vector Classifier': Pipeline([
        ('scale', StandardScaler()),
        ('model', SVC(kernel='rbf', C=1.0, gamma='scale', probability=True,
                     random_state=RANDOM_SEED)),
    ]),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
scoring = {
    'accuracy': 'accuracy',
    'balanced_accuracy': 'balanced_accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1',
    'roc_auc': 'roc_auc',
}

print(f'Training households (80%): {len(X_cls_train):,}')
print(f'Testing households (20%):  {len(X_cls_test):,}')
print(f'Five CV validation folds:   approximately {len(X_cls_train) // 5:,} households each')


### Five-fold cross-validation

Accuracy is reported for direct comparison with the assignment requirement, while balanced accuracy, precision, recall, F1, and ROC-AUC provide a more complete assessment. F1 is used as the primary selection metric because it rewards a useful balance between correctly identifying higher-spending households and limiting false higher-spend predictions. Scaling is included inside the KNN and SVC pipelines, so each validation fold is transformed using statistics learned only from its corresponding training folds.


In [ ]:
cv_rows = []

for model_name, model in classifiers.items():
    scores = cross_validate(
        model,
        X_cls_train,
        y_cls_train,
        cv=cv,
        scoring=scoring,
        n_jobs=1,
    )
    row = {'model': model_name}
    for metric in scoring:
        row[f'{metric}_mean'] = scores[f'test_{metric}'].mean()
        row[f'{metric}_std'] = scores[f'test_{metric}'].std()
    cv_rows.append(row)

cv_results = pd.DataFrame(cv_rows).sort_values('f1_mean', ascending=False)
display(cv_results.round(3))

cv_plot = cv_results.melt(
    id_vars='model',
    value_vars=['accuracy_mean', 'precision_mean', 'recall_mean', 'f1_mean', 'roc_auc_mean'],
    var_name='metric',
    value_name='mean_score',
)
cv_plot['metric'] = cv_plot['metric'].str.replace('_mean', '', regex=False).str.upper()

plt.figure(figsize=(12, 5))
sns.barplot(data=cv_plot, x='metric', y='mean_score', hue='model')
plt.ylim(0.5, 1.0)
plt.title('Five-Fold Cross-Validation Performance (80% Training Partition)')
plt.xlabel('Evaluation Metric')
plt.ylabel('Mean Validation Score')
plt.legend(title='Classifier', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()


### Held-out test performance and visual comparison

After cross-validation, each classifier is refitted on the full 80% training partition and evaluated once on the untouched 20% test partition. The confusion matrices show the types of errors, and the ROC curves compare discrimination across all probability thresholds rather than at only the default classification threshold.


In [ ]:
fitted_classifiers = {}
test_rows = []

for model_name, model in classifiers.items():
    fitted_model = clone(model).fit(X_cls_train, y_cls_train)
    fitted_classifiers[model_name] = fitted_model
    predictions = fitted_model.predict(X_cls_test)
    probabilities = fitted_model.predict_proba(X_cls_test)[:, 1]
    test_rows.append({
        'model': model_name,
        'accuracy': accuracy_score(y_cls_test, predictions),
        'balanced_accuracy': balanced_accuracy_score(y_cls_test, predictions),
        'precision': precision_score(y_cls_test, predictions),
        'recall': recall_score(y_cls_test, predictions),
        'f1': f1_score(y_cls_test, predictions),
        'roc_auc': roc_auc_score(y_cls_test, probabilities),
    })

test_results = pd.DataFrame(test_rows).sort_values('f1', ascending=False)
display(test_results.round(3))

figure, axes = plt.subplots(1, 3, figsize=(15, 4))
for axis, (model_name, fitted_model) in zip(axes, fitted_classifiers.items()):
    ConfusionMatrixDisplay.from_estimator(
        fitted_model,
        X_cls_test,
        y_cls_test,
        display_labels=['Lower Spend', 'Higher Spend'],
        cmap='Blues',
        colorbar=False,
        ax=axis,
    )
    axis.set_title(model_name)
figure.suptitle('Held-Out Test Confusion Matrices', fontsize=14, y=1.04)
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 6))
roc_axis = plt.gca()
for model_name, fitted_model in fitted_classifiers.items():
    RocCurveDisplay.from_estimator(
        fitted_model,
        X_cls_test,
        y_cls_test,
        name=model_name,
        ax=roc_axis,
    )
plt.plot([0, 1], [0, 1], linestyle='--', color='grey', label='Random classifier')
plt.title('Held-Out Test ROC Curves')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()


### Discussion of classification results

The three methods provide a useful contrast. Gaussian Naive Bayes is a simple probabilistic baseline and assumes conditional independence between predictors. K-Nearest Neighbours is a local, distance-based method that can represent irregular boundaries but is sensitive to nearby observations. The RBF Support Vector Classifier can learn a smooth non-linear boundary and is well suited to the scaled behavioural attributes.

The final model is selected by mean cross-validated F1 rather than by test performance, preventing the test set from influencing model choice. **RBF Support Vector Classifier was the most successful model**: it achieved mean five-fold F1 and accuracy of **0.930**, ahead of KNN (test F1 **0.906**) and Gaussian Naive Bayes (test F1 **0.861**). Its held-out F1 of **0.930**, accuracy of **0.929**, and ROC-AUC of **0.986** closely agree with cross-validation, providing evidence that the result generalizes beyond the training folds.

SVC likely performed best because household spending behaviour has a smooth but non-linear boundary that is not captured by Naive Bayes' conditional-independence assumption. It also learns a more stable global boundary than KNN, whose decisions depend on local neighbourhoods. The remaining errors are reasonable because the model uses only four leakage-safe behavioural indicators and never sees the `total_spend` value that defines the outcome.


In [ ]:
best_model_name = cv_results.iloc[0]['model']
best_cv = cv_results.set_index('model').loc[best_model_name]
best_test = test_results.set_index('model').loc[best_model_name]

print(f'Selected classifier: {best_model_name}')
print(f'Mean 5-fold CV F1:   {best_cv["f1_mean"]:.3f} '
      f'(SD {best_cv["f1_std"]:.3f})')
print(f'Mean CV accuracy:    {best_cv["accuracy_mean"]:.3f}')
print(f'Held-out test F1:    {best_test["f1"]:.3f}')
print(f'Held-out accuracy:   {best_test["accuracy"]:.3f}')
print(f'Held-out ROC-AUC:    {best_test["roc_auc"]:.3f}')

print(
    '\nInterpretation: The selected model achieved the strongest average balance '
    'of precision and recall across the five training folds. Its held-out scores '
    'show how well that choice generalizes to unseen households. The remaining '
    'errors are plausible because the four leakage-safe shopping behaviours do '
    'not encode total spending directly; they provide useful signals without '
    'making the prediction artificially easy.'
)
